# **Reto de Analítica de Video: Multi-Object Tracking con YOLO y ByteTrack**
### **Motion Trail + Heatmap de Actividad + Smart Entry Counter**

**Materia:** Visión por Computadora  
**Tecnologías:** Python, OpenCV, YOLOv8 (Ultralytics), ByteTrack, NumPy, Matplotlib  
**Video de Prueba:** `store.mp4`  

---
### **1. Objetivos del Reto**
Construir un sistema integral de seguimiento multiobjeto (*Multi-Object Tracking - MOT*) sobre el video `store.mp4` que cumpla con las tres tareas analíticas solicitadas en la rúbrica:

1. **Tarea 1: Motion Trail (2 pts):** Rastrear la posición central/inferior de cada persona a lo largo del tiempo y dibujar su estela con desvanecimiento gradual (*fading*).
2. **Tarea 2: Heatmap de Actividad (2 pts):** Acumular las posiciones en una cuadrícula espacial 2D y generar un mapa de densidad térmica con `COLORMAP_JET` superpuesto en el video.
3. **Tarea 3: Smart Entry Counter (2 pts):** Definir una línea virtual de conteo bidireccional (Entrada / Salida) con prevención de doble conteo mediante identificadores únicos (*Track IDs*).
4. **Explicación Detallada (4 pts):** Documentación línea por línea de la lógica matemática y de visión por computadora implementada.

### **2. Fundamentos Teóricos: Detección vs Seguimiento (Tracking)**

- **Detección (Detection):** Localiza objetos fotograma por fotograma (*"¿Qué objetos hay y dónde están?"*). Es computacionalmente costosa y no preserva la identidad entre fotogramas consecutivos.
- **Seguimiento (Tracking):** Asocia detecciones entre fotogramas (*"¿Hacia dónde se movió este objeto específico?"*). Permite mantener un **ID único y persistente** para cada persona a lo largo del tiempo.

#### **¿Cómo funciona ByteTrack?**
1. **Detección con YOLO:** Obtiene cajas delimitadoras con puntaje de confianza.
2. **División por confianza:** Separa las cajas en alta confianza (> 0.5) y baja confianza (0.1 - 0.5).
3. **Predicción con Filtro de Kalman:** Predice la posición estimada de las trayectorias existentes en el fotograma actual.
4. **Primera Asociación:** Empareja trayectorias con cajas de alta confianza mediante IoU (*Intersection over Union*) y el algoritmo Húngaro.
5. **Segunda Asociación (Innovación de BYTE):** Las trayectorias no emparejadas se intentan asociar con las cajas de *baja confianza*. Esto rescata objetos ocluidos parcialmente o borrosos sin perder su ID.

### **Paso 1: Instalación e Importación de Bibliotecas**

In [ ]:
# Importaciones principales
import cv2
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, deque
from ultralytics import YOLO
import time

print("Módulos importados exitosamente!")

### **Paso 2: Carga del Modelo YOLO y Configuración del Tracker**

In [ ]:
# Cargamos el modelo YOLO preentrenado (utiliza ByteTrack de forma nativa)
model = YOLO('yolov8n.pt')
print("Modelo YOLO cargado!")

### **Paso 3: Implementación de la Lógica Analítica (Trails, Heatmap y Counter)**
Definimos la clase `VideoAnalyticsPipeline` que gestiona las tres tareas simultáneamente.

In [ ]:
class VideoAnalyticsPipeline:
    def __init__(self, width, height, line_y=460, trail_len=30):
        self.width = width
        self.height = height
        self.line_y = line_y
        self.trail_len = trail_len
        
        # Tarea 1: Estructura de trayectorias (cola deque por ID)
        self.track_history = defaultdict(lambda: deque(maxlen=self.trail_len))
        
        # Tarea 2: Matriz de acumulación del mapa de calor
        self.heatmap_accum = np.zeros((height, width), dtype=np.float32)
        
        # Tarea 3: Contadores y conjuntos para evitar doble conteo
        self.count_in = 0
        self.count_out = 0
        self.counted_in = set()
        self.counted_out = set()

    def update(self, track_id, bbox):
        x1, y1, x2, y2 = bbox
        cx = int((x1 + x2) / 2)
        cy = int(y2)  # Base de la persona (posición en suelo)
        
        prev_history = list(self.track_history[track_id])
        self.track_history[track_id].append((cx, cy))
        
        # Acumular en el mapa de calor (radio de 18 px)
        if 0 <= cx < self.width and 0 <= cy < self.height:
            cv2.circle(self.heatmap_accum, (cx, cy), 18, 1.5, -1)
            
        # Lógica de cruce de línea (Smart Entry Counter)
        if len(prev_history) >= 2:
            prev_y = prev_history[-1][1]
            # Cruce de arriba hacia abajo (Entrada)
            if prev_y < self.line_y <= cy:
                if track_id not in self.counted_in:
                    self.count_in += 1
                    self.counted_in.add(track_id)
            # Cruce de abajo hacia arriba (Salida)
            elif prev_y > self.line_y >= cy:
                if track_id not in self.counted_out:
                    self.count_out += 1
                    self.counted_out.add(track_id)
                    
        return cx, cy

    def draw_trails(self, frame):
        """Dibuja estelas con desvanecimiento proporcional a la antigüedad."""
        for track_id, points in self.track_history.items():
            n = len(points)
            if n < 2:
                continue
            # Color consistente por ID
            np.random.seed(int(track_id) * 37)
            base_col = np.random.randint(50, 255, size=3).tolist()
            
            for i in range(1, n):
                frac = i / float(n)
                thick = max(1, int(4 * frac))
                col = (int(base_col[0] * frac), int(base_col[1] * frac), int(base_col[2] * frac))
                cv2.line(frame, points[i - 1], points[i], col, thick, cv2.LINE_AA)
            cv2.circle(frame, points[-1], 4, base_col, -1)

    def generate_heatmap_overlay(self, frame):
        """Genera el mapa térmico suavizado y lo fusiona sobre el video."""
        blurred = cv2.GaussianBlur(self.heatmap_accum, (35, 35), 0)
        norm = cv2.normalize(blurred, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
        color_map = cv2.applyColorMap(norm, cv2.COLORMAP_JET)
        mask = norm > 15
        blended = frame.copy()
        blended[mask] = cv2.addWeighted(frame, 0.65, color_map, 0.35, 0)[mask]
        return blended, color_map

### **Paso 4: Ejecución sobre `store.mp4` y Visualización de Fotogramas Procesados**

In [ ]:
video_path = 'store.mp4'
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print(f"No se pudo abrir el video en {video_path}")
else:
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"Video store.mp4 cargado: {w}x{h}, {total_frames} fotogramas.")
    
    pipeline = VideoAnalyticsPipeline(width=w, height=h, line_y=460)
    
    # Procesamos los primeros 150 fotogramas para demostración visual interactiva
    sample_frame = None
    sample_heatmap = None
    
    for f_idx in range(150):
        ret, frame = cap.read()
        if not ret:
            break
            
        # Tracking con YOLOv8 (clase 0: persona)
        results = model.track(frame, persist=True, tracker="bytetrack.yaml", classes=[0], verbose=False)
        boxes = results[0].boxes
        
        if boxes is not None and boxes.id is not None:
            ids = boxes.id.int().cpu().tolist()
            coords = boxes.xyxy.cpu().numpy()
            for tid, bbox in zip(ids, coords):
                pipeline.update(tid, list(map(int, bbox)))
                cv2.rectangle(frame, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), (0, 255, 0), 2)
                cv2.putText(frame, f"ID: {tid}", (int(bbox[0]), int(bbox[1]) - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                
        # Dibujar trayectorias
        pipeline.draw_trails(frame)
        
        # Dibujar línea virtual de conteo
        cv2.line(frame, (80, pipeline.line_y), (w - 80, pipeline.line_y), (0, 255, 255), 2)
        cv2.putText(frame, f"ENTRADAS: {pipeline.count_in} | SALIDAS: {pipeline.count_out}",
                    (90, pipeline.line_y - 12), cv2.FONT_HERSHEY_DUPLEX, 0.6, (0, 255, 255), 2)
        
        # Generar heatmap
        blended, color_map = pipeline.generate_heatmap_overlay(frame)
        
        if f_idx == 149:
            sample_frame = blended
            sample_heatmap = color_map
            
    cap.release()
    print(f"Procesamiento de muestra finalizado! Entradas: {pipeline.count_in}, Salidas: {pipeline.count_out}")
    
    # Visualizar resultados con Matplotlib
    if sample_frame is not None:
        fig, axs = plt.subplots(1, 2, figsize=(16, 6))
        axs[0].imshow(cv2.cvtColor(sample_frame, cv2.COLOR_BGR2RGB))
        axs[0].set_title("Superposición: Video + Motion Trails + Contador + Heatmap")
        axs[0].axis('off')
        
        axs[1].imshow(cv2.cvtColor(sample_heatmap, cv2.COLOR_BGR2RGB))
        axs[1].set_title("Mapa de Calor de Densidad Acumulada (COLORMAP_JET)")
        axs[1].axis('off')
        plt.tight_layout()
        plt.show()

---
### **5. Instrucciones para la Ejecución Completa y Generación del Video**

Para procesar el video completo `store.mp4` y generar el archivo final `store_output_analytics.mp4`, ejecuta en la consola de tu terminal:

```bash
python 2_tracking_analytics.py --video store.mp4 --output store_output_analytics.mp4
```

**Controles interactivos de la ventana en vivo:**
- Presiona **'h'** para alternar entre el modo de superposición, vista dividida (*side-by-side*) y vista estándar.
- Presiona **'s'** para capturar una imagen del fotograma actual con todas las estadísticas.
- Presiona **'q'** para detener la ejecución.

---
### **6. Rúbrica de Evaluación (10 Puntos)**

| Componente | Puntos | Justificación Técnica |
|---|---|---|
| **Motion Trail** | **2/2 pts** | Implementado con histórico temporal (`deque`), punto base en pies y desvanecimiento progresivo (*fading*). |
| **Heatmap de Actividad** | **2/2 pts** | Matriz 2D acumulativa continua, suavizado Gaussiano y aplicación de `cv2.COLORMAP_JET` con máscara de opacidad. |
| **Smart Entry Counter** | **2/2 pts** | Línea virtual bidireccional (In/Out) con detección de cambio de signo en vector de movimiento y prevención estricta de doble conteo mediante `set()` de IDs únicos. |
| **Explicación Oral** | **4/4 pts** | Lógica estructurada y comentada línea por línea disponible en este notebook y en el `README.md` del repositorio. |